# EDA2 · Overtime persistence across the Metropolitan health services

**Course:** Business Intelligence · Universidad del Desarrollo · W2 supporting notebook
**Group 01:** Alessandro Lavezzi · José Saavedra

**Purpose.** Apply exactly the same preparation and KPI definitions used for the Servicio de Salud Metropolitano Central (SSMC) in `EDA.ipynb` (W1/C1) and `W2.ipynb` to the other five Metropolitan health services (Norte, Oriente, Occidente, Sur, Sur Oriente). The comparison answers two questions that the SSMC data alone cannot:

1. Are the SSMC results (persistence rate, concentration in auxiliary and administrative staff, decline between years) **specific to the SSMC** or common to the network?
2. How well does the SSMC represent fixed-term staff of public health services more broadly? (input for W2, Section D: coverage and representativeness)

**Source.** State Transparency Portal (Active Transparency), fixed-term (*contrata*) staff payroll, one CSV per service and month (`Data/<Service>_TransparenciaActiva_YYYY-MM[_Establishment].csv`; the SSMC files have no prefix). The Sur service publishes one file per establishment from January 2025.

**Run order.** Kernel → Restart & Run All. The notebook only reads `../Data/` and writes figures `fig_eda2_*.png` to the same folder. No randomness is used.

**Labels used in the text.** *Observation* = what the output shows. *Assumption* = a choice the team makes that the data cannot verify. *Limitation* = what the output cannot establish.

In [ ]:
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: "%.3f" % x)

## 1. Configuration, loading and integration

- `SERVICES`: file pattern of each service. The SSMC (`Centro`) has no prefix, as in `EDA.ipynb`.
- The period is read from the file name with a regular expression (`YYYY-MM`). *Correction:* the first draft of this notebook took the last `_`-separated token as the period, so the Sur establishment files received periods such as `ServicioSur`, the common window collapsed to two months and the persistence KPI could not be computed (see the W2 feedback table, entry F3).
- `KNOWN_ATYPICAL`: months marked as atypical before this notebook (SSMC January 2025, C1 §4.3). Other atypical months are **detected by a rule** in Section 3.2, not chosen by eye.

In [ ]:
#CONFIGURATION
DATA_DIR = Path("..") / "Data"

SERVICES = {                                   # file pattern of each Metropolitan health service
    "Centro": "TransparenciaActiva_*.csv",
    "Norte": "Norte_TransparenciaActiva_*.csv",
    "Oriente": "Oriente_TransparenciaActiva_*.csv",
    "Occidente": "Occidente_TransparenciaActiva_*.csv",
    "Sur": "Sur_TransparenciaActiva_*.csv",
    "SurOriente": "SurOriente_TransparenciaActiva_*.csv",
}
ESTABLISHMENT_ALIASES = {"HOSPITALDRLUCIOCORDOVA": "HospitalLucioCordova"}   # same hospital, two file names

PERSISTENCE_THRESHOLD = 75                     # % of payroll months with overtime
MIN_MONTHS = 12                                # months in payroll to enter the persistence denominator
KNOWN_ATYPICAL = {"Centro": ["2025-01"]}       # C1 §4.3: different reporting format in the SSMC
ROBUST_Z_LIMIT = 3.5                           # |robust z| above this marks a month as atypical (Section 3.2)
MIN_ROWS_SHARE = 0.90                          # a month with < 90% of the service's median person-months is incomplete
YEAR1 = ("2024-11", "2025-08")                 # same calendar months in both years (Section 6)
YEAR2 = ("2025-11", "2026-08")

STAFF_GROUP_EN = {
    "Auxiliar": "Auxiliary", "Administrativo": "Administrative", "Técnico": "Technician",
    "Profesional": "Professional", "Directivo": "Executive",
    "Médicos cirujanos, farmacéuticos, químico-farmacéuticos, bioquímicos, cirujano-dentistas": "Physicians & related",
}
GROUP_ORDER = ["Auxiliary", "Administrative", "Technician", "Professional", "Physicians & related"]
COLOR = dict(zip(SERVICES, plt.cm.tab10.colors))   # fixed colour per service in every figure

In [ ]:
#LOADING ALL SERVICES
RAW_NAMES = [
    "year", "month_name", "staff_group", "name", "position", "grade", "qualification", "region",
    "special_allowances", "gross_pay_txt", "net_pay_txt", "additional_pay_txt", "bonus_txt",
    "ot_entitled", "ot_day_txt", "ot_night_txt", "ot_holiday_txt", "start_date_txt", "end_date_txt",
    "travel_allowance", "remarks",
]
PERIOD_RE = re.compile(r"\d{4}-\d{2}")

frames, files_log, reference_header = [], [], None
for service, pattern in SERVICES.items():
    for f in sorted(DATA_DIR.glob(pattern)):
        t = pd.read_csv(f, sep=";", encoding="latin-1", dtype=str)
        period = PERIOD_RE.search(f.stem).group()
        suffix = f.stem.split(period)[-1].strip("_")
        unnamed = [c for c in t.columns if c.startswith("Unnamed")]
        header = [c for c in t.columns if c not in unnamed]
        reference_header = reference_header or header
        if header != reference_header:
            raise ValueError(f"{f.name}: header differs from the reference file; renaming by position is unsafe")
        files_log.append({"service": service, "file": f.name, "period": period,
                          "establishment": ESTABLISHMENT_ALIASES.get(suffix, suffix) or service,
                          "rows": len(t), "unnamed_empty": bool(t[unnamed].isna().all().all())})
        t = t.drop(columns=unnamed)
        t.columns = RAW_NAMES
        t.insert(0, "service", service)
        t["period"] = period
        t["establishment"] = files_log[-1]["establishment"]
        frames.append(t)

raw = pd.concat(frames, ignore_index=True)
files_log = pd.DataFrame(files_log)
print("Files read:", len(files_log), "| rows:", len(raw), "| sum of file rows:", files_log["rows"].sum())
print("Trailing unnamed column empty in every file:", files_log["unnamed_empty"].all())

In [ ]:
#TYPE CONVERSION (SAME RULES AS W1 §2 AND W2)
def to_amount(s):
    # "$ 1.234.567" -> 1234567; a decimal comma ("1.303.067,00", Norte) is kept as a decimal point
    clean = (s.astype(str).str.replace("$", "", regex=False).str.strip()
              .str.replace(".", "", regex=False).str.replace(",", ".", regex=False))
    return pd.to_numeric(clean, errors="coerce").astype(float)

def split_overtime(s):
    txt = s.str.strip()
    none = txt.eq("No tiene")                                      # "No tiene" = no overtime of that type -> 0
    parts = txt.str.split(":", n=1, expand=True).reindex(columns=[0, 1])
    amount = to_amount(parts[0])
    hours = pd.to_numeric(parts[1].astype(str).str.replace("hrs", "", regex=False)
                                   .str.replace(".", "", regex=False)
                                   .str.replace(",", ".", regex=False).str.strip(),
                          errors="coerce").astype(float)
    return amount.where(~none, 0.0), hours.where(~none, 0.0)

payroll = raw.copy()
for txt_col, col in {"gross_pay_txt": "gross_pay", "net_pay_txt": "net_pay",
                     "additional_pay_txt": "additional_pay", "bonus_txt": "bonus"}.items():
    payroll[col] = to_amount(payroll[txt_col])
for kind in ["day", "night", "holiday"]:
    payroll[f"ot_{kind}_amount"], payroll[f"ot_{kind}_hours"] = split_overtime(payroll[f"ot_{kind}_txt"])
for txt_col, col in {"start_date_txt": "start_date", "end_date_txt": "end_date"}.items():
    payroll[col] = pd.to_datetime(payroll[txt_col], format="%d/%m/%Y", errors="coerce")

payroll["ot_amount"] = payroll[["ot_day_amount", "ot_night_amount"]].sum(axis=1)
payroll["ot_hours"] = payroll[["ot_day_hours", "ot_night_hours"]].sum(axis=1)
payroll["position_norm"] = payroll["position"].str.strip().str.upper().replace({"CHOFER": "CONDUCTOR"})
payroll["staff_group_en"] = payroll["staff_group"].str.strip().map(STAFF_GROUP_EN)
payroll["person_id"] = pd.factorize(payroll["service"] + "|" + payroll["name"].str.strip())[0] + 1   # names are not displayed

coverage = files_log.groupby("service").agg(files=("file", "size"), establishments=("establishment", "nunique"),
                                            first=("period", "min"), last=("period", "max"),
                                            months=("period", "nunique"), rows=("rows", "sum")).reindex(SERVICES)
coverage["persons"] = payroll.groupby("service")["person_id"].nunique()
coverage

In [ ]:
#ROWS PER MONTH AND SERVICE (CONTRACT RECORDS)
files_log.pivot_table(index="period", columns="service", values="rows", aggfunc="sum").reindex(columns=SERVICES)

### 1.1 Common window

Services do not cover the same months: the SSMC ends in August 2026, the others in September 2026, and Sur starts in November 2024. KPIs are only comparable over the months present in **every** service, so all comparisons use the common window. Rows outside it are not deleted from `payroll`; they are only excluded from `payroll_w`, and the count is reported.

*Assumption:* 22 months are enough to apply the 12-month persistence denominator. The SSMC values over its own 24 months are reproduced separately in Section 3 to check that the code matches C1.

In [ ]:
#COMMON WINDOW
periods_by_service = payroll.groupby("service")["period"].apply(set)
WINDOW = sorted(set.intersection(*periods_by_service))
all_periods = sorted(payroll["period"].unique())
assert WINDOW == [p for p in all_periods if WINDOW[0] <= p <= WINDOW[-1]], "the common window has gaps"
print(f"Common window: {WINDOW[0]} to {WINDOW[-1]} ({len(WINDOW)} months)")

outside = payroll[~payroll["period"].isin(WINDOW)]
print("Rows outside the common window (kept in `payroll`, excluded from `payroll_w`):")
display(outside.pivot_table(index="period", columns="service", values="name", aggfunc="size")
               .reindex(columns=SERVICES).fillna(0).astype(int))

payroll_w = payroll[payroll["period"].isin(WINDOW)].copy()
print("Rows before:", len(payroll), "| after:", len(payroll_w), "| outside the window:", len(payroll) - len(payroll_w))

## 2. Data-quality checks by service

The SSMC checks of `EDA.ipynb` and W2 Section C are repeated in every service, to detect recording problems specific to each one **before** comparing KPIs. Each violation is a *flag*, not an automatic deletion (Class 4: "a violation is a flag, not an automatic treatment decision").

1. Values that cannot be converted (amounts, overtime, dates), separating coded values (`-`, `Indefinido`) from true failures.
2. Monthly share of `-` in additional pay and bonuses (missingness mechanism).
3. Exact duplicates and persons with more than one record in a month.
4. Staff-group changes between consecutive months.
5. Holiday overtime, overtime paid without entitlement, zero gross pay, overtime above gross pay, staff-group values.
6. The same name in more than one service.

In [ ]:
#2.1 CONVERSION FAILURES AND CODED VALUES BY SERVICE
svc = payroll_w["service"]
conv = {}
for txt_col, col in {"gross_pay_txt": "gross_pay", "net_pay_txt": "net_pay", "additional_pay_txt": "additional_pay",
                     "bonus_txt": "bonus", "start_date_txt": "start_date", "end_date_txt": "end_date"}.items():
    txt = payroll_w[txt_col].str.strip()
    coded = txt.isin(["-", "Indefinido"])
    conv[f"{col}: coded"] = coded.groupby(svc).sum()
    conv[f"{col}: not convertible"] = (payroll_w[col].isna() & txt.notna() & ~coded).groupby(svc).sum()
for kind in ["day", "night", "holiday"]:
    bad = (payroll_w[f"ot_{kind}_amount"].isna() | payroll_w[f"ot_{kind}_hours"].isna()) & payroll_w[f"ot_{kind}_txt"].notna()
    conv[f"ot_{kind}: not convertible"] = bad.groupby(svc).sum()
conv = pd.DataFrame(conv).reindex(SERVICES).T
conv

In [ ]:
#2.2 SHARE OF "-" BY MONTH AND SERVICE (%)
for col in ["additional_pay_txt", "bonus_txt"]:
    print(col.replace("_txt", ""), '- % of contract rows with "-"')
    display((payroll_w[col].str.strip().eq("-").groupby([payroll_w["period"], svc]).mean().unstack()
             .reindex(columns=SERVICES) * 100).round(0))

In [ ]:
#2.3 DUPLICATES AND PERSON-MONTH KEY
key_cols = ["service", "period"] + RAW_NAMES
exact_dup = payroll_w.duplicated(subset=key_cols)
records_per_key = payroll_w.groupby(["service", "period", "person_id"]).size()

dup = pd.DataFrame({
    "rows": svc.value_counts(),
    "exact_duplicate_copies": exact_dup.groupby(svc).sum(),
    "duplicates_with_overtime": (exact_dup & (payroll_w["ot_hours"] > 0)).groupby(svc).sum(),
    "person_months": records_per_key.groupby("service").size(),
    "person_months_with_>1_record": (records_per_key > 1).groupby("service").sum(),
    "max_records_in_a_person_month": records_per_key.groupby("service").max(),
}).reindex(SERVICES)
dup["extra_rows_%"] = (dup["rows"] - dup["person_months"]) / dup["rows"] * 100
dup.round(1)

In [ ]:
#2.4 STAFF-GROUP CHANGES BETWEEN CONSECUTIVE MONTHS
sg = (payroll_w.groupby(["service", "person_id", "period"])["staff_group_en"].first().reset_index()
               .sort_values(["service", "person_id", "period"]))
sg["previous"] = sg.groupby(["service", "person_id"])["staff_group_en"].shift()
changes = sg[sg["previous"].notna() & (sg["staff_group_en"] != sg["previous"])]
changes_month = (changes.pivot_table(index="period", columns="service", values="person_id", aggfunc="size")
                        .reindex(index=WINDOW, columns=SERVICES).fillna(0).astype(int))
changes_month.T

In [ ]:
#2.5 OTHER CHECKS BY SERVICE
has_ot_row = payroll_w["ot_hours"] > 0
other = pd.DataFrame({
    "rows_with_holiday_ot": (payroll_w["ot_holiday_hours"] > 0).groupby(svc).sum(),
    "ot_paid_without_entitlement": (has_ot_row & payroll_w["ot_entitled"].str.strip().eq("No")).groupby(svc).sum(),
    "zero_gross_pay": (payroll_w["gross_pay"] == 0).groupby(svc).sum(),
    "ot_amount_above_gross_pay": (payroll_w["ot_amount"] > payroll_w["gross_pay"]).groupby(svc).sum(),
    "distinct_regions": payroll_w.groupby("service")["region"].nunique(),
    "positions_raw": payroll_w.groupby("service")["position"].nunique(),
    "positions_normalised": payroll_w.groupby("service")["position_norm"].nunique(),
    "staff_group_unmapped": payroll_w["staff_group_en"].isna().groupby(svc).sum(),
    "executive_rows": payroll_w["staff_group_en"].eq("Executive").groupby(svc).sum(),
}).reindex(SERVICES)
other

In [ ]:
#2.6 THE SAME NAME IN MORE THAN ONE SERVICE
services_per_name = payroll_w.groupby(payroll_w["name"].str.strip())["service"].nunique()
same_month = payroll_w.groupby(["period", payroll_w["name"].str.strip()])["service"].nunique()
print("Names present in more than one service (any month):", (services_per_name > 1).sum(),
      "of", len(services_per_name))
print("Name-months present in more than one service in the same month:", (same_month > 1).sum())

**Observations (Section 2)**

- *Conversion (2.1).* After coding `-` (not reported) and `Indefinido` (open-ended contract, 354 Occidente end dates) as missing values, **no value fails conversion in any service**. The two Norte night-overtime amounts written with a decimal comma (`1.303.067,00`) were lost by the W1 parser; the parser now reads them.
- *Missingness mechanism (2.2).* The `-` pattern is a property of **the service and the month**, not of the person: Sur Oriente never reports additional pay or bonuses (100% `-` every month), Norte has a stable ~25–30% `-`, Oriente switches between 0% and 100%, and the SSMC has whole-month episodes (additional pay in January 2025; bonuses in May 2025 and July 2026). *Decision:* bonuses and additional pay are not compared across services and are not used in any KPI.
- *Duplicates and key (2.3).* Exact duplicate rows exist only in the SSMC (27, none with overtime, C1 §6.1). All services have persons with more than one record in a month (0.8%–6.9% extra rows), so the person-month aggregation is needed everywhere.
- *Staff-group changes (2.4).* The SSMC shows 218 changes in December 2025 and 213 in January 2026 (the recording error and its reversal, C1); every other service shows 0–11 changes per month. *Decision:* modal staff group per person, as in C1.
- *Other checks (2.5).* **Holiday overtime is SSMC-specific in being empty**: Sur Oriente has 7,252 contract rows with holiday overtime, Oriente 1,557 and Norte 378. The KPIs keep the C1 definition (day + night hours) so that the SSMC is measured exactly as before; *limitation:* this understates overtime in Sur Oriente in particular. Overtime paid without declared entitlement is frequent in Oriente (988 rows), so the entitlement field is not a reliable filter. Zero gross pay (2,004 rows) and overtime above gross pay (302) concentrate in Occidente, which makes its weight KPI less reliable (the median is used). Norte records 765 distinct position labels against 80–212 elsewhere, so **positions cannot be compared across services**; position analysis stays within the SSMC. `Directivo` (Executive) appears in 15 rows and was missing from the C1 dictionary.
- *Identity (2.6).* 129 of 16,433 names appear in more than one service (109 name-months simultaneously). Without RUT they are counted as different persons; they are under 1% of persons, so the effect on rates is negligible.

## 3. Person-month table and KPIs by service

The same person-month table as C1/W2 is built with key **(service, period, person)**. A person is identified by **(service, name)**: the same name in two services counts as two persons, because there is no national ID (RUT) to link them (check 2.6).

- Staff group: the most frequent value of each person within the service and window (as `staff_group_modal` in C1).
- Flags (marked, never deleted): `flag_zero_pay`, `flag_impossible` (overtime amount > gross pay), `flag_extreme_hours` (above Q3 + 1.5·IQR **within each service**), `flag_atypical_month` and `flag_amount_month` (Section 3.2).

In [ ]:
#3.1 PERSON-MONTH TABLE FOR ALL SERVICES
def build_person_months(p):
    out = (p.groupby(["service", "period", "person_id"])
            .agg(staff_group=("staff_group_en", "first"), position=("position_norm", "first"),
                 day_hours=("ot_day_hours", "sum"), night_hours=("ot_night_hours", "sum"),
                 day_amount=("ot_day_amount", "sum"), night_amount=("ot_night_amount", "sum"),
                 gross_pay=("gross_pay", "sum"))
            .reset_index())
    out["ot_hours"] = out["day_hours"] + out["night_hours"]
    out["ot_amount"] = out["day_amount"] + out["night_amount"]
    out["has_ot"] = out["ot_hours"] > 0
    out["over_40_day"] = out["day_hours"] > 40
    out["ot_weight"] = np.where(out["gross_pay"] > 0, out["ot_amount"] / out["gross_pay"] * 100, np.nan)
    modal = p.groupby("person_id")["staff_group_en"].agg(lambda s: s.mode().iloc[0])
    out["staff_group_modal"] = out["person_id"].map(modal)
    return out

pm_all = build_person_months(payroll_w)

limits = pm_all[pm_all["has_ot"]].groupby("service")["ot_hours"].quantile([0.25, 0.75]).unstack()
limits["extreme_limit"] = limits[0.75] + 1.5 * (limits[0.75] - limits[0.25])
pm_all["flag_zero_pay"] = pm_all["gross_pay"] <= 0
pm_all["flag_impossible"] = pm_all["ot_weight"] > 100
pm_all["flag_extreme_hours"] = pm_all["ot_hours"] > pm_all["service"].map(limits["extreme_limit"])

print("Person-months:", len(pm_all), "| persons:", pm_all["person_id"].nunique())
print("Aggregation preserves totals by service:")
for a, b in [("ot_day_hours", "day_hours"), ("ot_night_hours", "night_hours"), ("ot_day_amount", "day_amount"),
             ("ot_night_amount", "night_amount"), ("gross_pay", "gross_pay")]:
    same = np.isclose(payroll_w.groupby("service")[a].sum(), pm_all.groupby("service")[b].sum()).all()
    print(f"  {b}: {same}")
print("Person-months without a staff group:", pm_all["staff_group_modal"].isna().sum())
print("Extreme-hours limit by service (h):")
limits.reindex(SERVICES).round(1)

### 3.2 Rule-based detection of atypical months

A month is compared with **the rest of its own service** using a robust z-score, $z = (x - \text{median}) / (1.4826 \cdot \text{MAD})$, which is not distorted by the outliers it is trying to find (Class 3: robust measures). Two kinds of flags are kept separate because they affect different KPIs:

| Flag | Rule | KPIs affected |
|---|---|---|
| `flag_atypical_month` | overtime frequency with \|z\| > 3.5, **or** fewer than 90% of the service's median person-months (incomplete file), **or** listed in `KNOWN_ATYPICAL` | frequency, persistence, weight |
| `flag_amount_month` | median amount paid per overtime hour with \|z\| > 3.5 (hours look normal, amounts do not) | weight only |

In [ ]:
#3.2 MONTHLY INDICATORS AND ROBUST Z-SCORES
with_ot = pm_all[pm_all["has_ot"]]
monthly = pd.DataFrame({
    "person_months": pm_all.groupby(["service", "period"]).size(),
    "ot_frequency_%": pm_all.groupby(["service", "period"])["has_ot"].mean() * 100,
    "amount_per_hour": (with_ot["ot_amount"] / with_ot["ot_hours"]).groupby([with_ot["service"], with_ot["period"]]).median(),
})

def robust_z(s):
    mad = (s - s.median()).abs().median() * 1.4826
    return (s - s.median()) / mad if mad > 0 else s * 0

for col in ["ot_frequency_%", "amount_per_hour"]:
    monthly[f"z_{col}"] = monthly.groupby(level="service")[col].transform(robust_z)
monthly["rows_share"] = monthly["person_months"] / monthly.groupby(level="service")["person_months"].transform("median")
known = pd.Series([p in KNOWN_ATYPICAL.get(s, []) for s, p in monthly.index], index=monthly.index)

monthly["reason"] = ""
monthly.loc[known, "reason"] += "known (C1); "
monthly.loc[monthly["z_ot_frequency_%"].abs() > ROBUST_Z_LIMIT, "reason"] += "frequency; "
monthly.loc[monthly["rows_share"] < MIN_ROWS_SHARE, "reason"] += "incomplete file; "
monthly.loc[monthly["z_amount_per_hour"].abs() > ROBUST_Z_LIMIT, "reason"] += "amount per hour; "
monthly["atypical"] = monthly["reason"].str.contains("known|frequency|incomplete")
monthly["amount_only"] = monthly["reason"].str.contains("amount") & ~monthly["atypical"]

flagged = monthly[monthly["reason"] != ""]
display(flagged[["person_months", "rows_share", "ot_frequency_%", "z_ot_frequency_%",
                 "amount_per_hour", "z_amount_per_hour", "reason"]].round(2))

ATYPICAL = {s: list(flagged[flagged["atypical"]].loc[s].index) if s in flagged[flagged["atypical"]].index else []
            for s in SERVICES}
AMOUNT_MONTHS = {s: list(flagged[flagged["amount_only"]].loc[s].index) if s in flagged[flagged["amount_only"]].index else []
                 for s in SERVICES}
key = list(zip(pm_all["service"], pm_all["period"]))
pm_all["flag_atypical_month"] = [p in ATYPICAL[s] for s, p in key]
pm_all["flag_amount_month"] = [p in AMOUNT_MONTHS[s] for s, p in key]
print("Atypical months:", {s: v for s, v in ATYPICAL.items() if v})
print("Amount-only months (weight not comparable):", {s: v for s, v in AMOUNT_MONTHS.items() if v})
print("Person-months flagged, by service:")
pm_all.groupby("service")[[c for c in pm_all.columns if c.startswith("flag_")]].sum().reindex(SERVICES)

In [ ]:
#FIGURE EDA2-1: MONTHLY QUALITY CONTROL (ROBUST Z BY SERVICE)
panels = {"Overtime frequency (robust z)": "z_ot_frequency_%",
          "Median amount per overtime hour (robust z)": "z_amount_per_hour"}
fig, axes = plt.subplots(1, 3, figsize=(16, 7), sharey=True)
for ax, (title, col) in zip(axes[:2], panels.items()):
    z = monthly[col].unstack(0).reindex(index=WINDOW, columns=SERVICES)
    sns.heatmap(z, cmap="RdBu_r", center=0, vmin=-6, vmax=6, annot=True, fmt=".1f", annot_kws={"size": 7},
                cbar=False, linewidths=0.5, linecolor="white", ax=ax)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Service"); ax.set_ylabel("Payment month")
share = monthly["rows_share"].unstack(0).reindex(index=WINDOW, columns=SERVICES) * 100
sns.heatmap(share, cmap="Greys_r", vmin=50, vmax=100, annot=True, fmt=".0f", annot_kws={"size": 7},
            cbar=False, linewidths=0.5, linecolor="white", ax=axes[2])
axes[2].set_title("Person-months as % of the service median", fontsize=10)
axes[2].set_xlabel("Service"); axes[2].set_ylabel("")
fig.suptitle(f"Figure EDA2-1. Monthly recording checks, {WINDOW[0]} to {WINDOW[-1]} "
             f"(|z| > {ROBUST_Z_LIMIT} or < {MIN_ROWS_SHARE:.0%} of median rows = flagged)")
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_eda2_1_monthly_checks.png", dpi=150)
plt.show()

**Observations (Section 3.2)**

- The rule flags **SSMC August 2025** on the amount per hour (z = −5.7) without being told about it: C1 found the same month by inspection (§8). This is evidence that the rule detects real recording changes.
- *Atypical months (frequency or coverage):* SSMC January 2025 (known from C1); **Occidente July and August 2026** (frequency 33.5% and 36.5% against a median near 24%, z = 6.6 and 8.8; July also has 116% of the usual person-months), compatible with a back-payment or a reporting change; **Sur October and November 2025** (64% and 87% of the usual person-months: the Hospital El Peral file is missing in October and several files are short in November).
- *Amount-only months (weight not comparable):* SSMC August 2025, Norte May 2025, Sur November 2024 and four Sur Oriente months. Their hours look normal, so they stay in frequency and persistence and are excluded only from the weight.
- *Assumption:* the 3.5 cut-off is the usual convention for robust z-scores; it is fixed before looking at the KPIs.
- *Limitation:* the rule compares a month with its own service, so a change that hits **all services in the same month** would not be flagged.

In [ ]:
#3.3 KPIs BY SERVICE (COMMON WINDOW)
def compute_kpis(t, threshold=PERSISTENCE_THRESHOLD):
    per = t.groupby("person_id").agg(months=("period", "nunique"), ot_months=("has_ot", "sum"))
    base = per[per["months"] >= MIN_MONTHS]
    persistent = base["ot_months"] / base["months"] * 100 >= threshold
    weight = t.loc[t["has_ot"] & (t["gross_pay"] > 0), "ot_weight"]
    with_day = (t["day_hours"] > 0).sum()
    return pd.Series({
        "person_months": len(t),
        "persons": len(per),
        "persons_12m": len(base),
        "persistent": persistent.sum(),
        "persistence_rate_%": persistent.mean() * 100,
        "ot_frequency_%": t["has_ot"].mean() * 100,
        "median_ot_weight_%": weight.median(),
        "median_hours_if_ot": t.loc[t["has_ot"], "ot_hours"].median(),
        "over_40_%_of_day_ot": t["over_40_day"].sum() / with_day * 100 if with_day else np.nan,
        "ot_spend_CLP_millions": t["ot_amount"].sum() / 1e6,
        "ot_spend_per_person_month": t["ot_amount"].sum() / len(t),
    })

def kpis_by_service(t, **kw):
    table = t.groupby("service")[t.columns].apply(lambda g: compute_kpis(g, **kw)).reindex(SERVICES)
    table.loc["All services"] = compute_kpis(t, **kw)
    return table

kpi_table = kpis_by_service(pm_all)
kpi_table.round(1)

In [ ]:
#3.4 VALIDATION: THE SSMC OVER ITS OWN 24 MONTHS MUST REPRODUCE C1
pm_centro_24 = build_person_months(payroll[payroll["service"] == "Centro"])
expected_c1 = {"person_months": 43120, "persons": 2430, "persons_12m": 1776, "persistent": 364}
got = compute_kpis(pm_centro_24)
for k, v in expected_c1.items():
    print(f"{k}: EDA2 = {got[k]:.0f} | C1 = {v} | match: {got[k] == v}")
assert all(got[k] == v for k, v in expected_c1.items()), "EDA2 does not reproduce the C1 SSMC values"
print(f"SSMC persistence rate: 24 months = {got['persistence_rate_%']:.1f}% | common window "
      f"({len(WINDOW)} months) = {kpi_table.loc['Centro', 'persistence_rate_%']:.1f}%")

**Observations (Sections 3.3–3.4)**

- *Reproduction:* over its own 24 months the code gives exactly the C1 SSMC values (43,120 person-months, 2,430 persons, 1,776 in the base, 364 persistent), so the comparison uses the same definitions as C1. Over the 22-month common window the SSMC rate is 20.1% (351 of 1,747), 0.4 points below the 24-month value.
- *Persistence:* the network rate is 16.7% (1,558 of 9,315 persons present ≥ 12 months). Services range from **7.6% (Oriente) to 22.7% (Sur)**; the SSMC is **3rd of 6**. Its level of persistence is above the network average but not exceptional.
- *Intensity:* the SSMC is at the top in every **intensity** measure: highest median overtime weight in gross pay (14.6%, next Sur 12.4%), highest median hours when overtime is paid (36 h, others 13–31 h), highest spending per person-month (CLP 78,816) and the highest share of person-months above 40 daytime hours (1.8%, others 0.1–0.6%; partly January 2025).
- *Frequency:* 27.2% of SSMC person-months have overtime, 3rd after Sur (30.3%) and close to Norte (26.3%).

## 4. Comparison by staff group

Persistence differs strongly between staff groups (C1 §7), so a difference in the overall rate between services could be caused only by **composition** (Simpson's paradox, Class 3). Three views:
- **Composition** of each service by staff group (segment proportions).
- **Persistence, frequency and weight** by staff group and service, with the number of persons behind each rate.
- **Directly standardised persistence rate**: the rate each service would have with the pooled staff-group composition of all services, $\text{rate}^{std}_s = \sum_g w_g \cdot \text{rate}_{s,g}$.

The 15 *Executive* contract rows (Sur and Sur Oriente) are too few for a rate and are left out of the staff-group tables, not out of the data.

In [ ]:
#4.1 PERSONS AND COMPOSITION BY STAFF GROUP
per_person = (pm_all.groupby(["service", "person_id"])
                    .agg(staff_group=("staff_group_modal", "first"), months=("period", "nunique"),
                         ot_months=("has_ot", "sum"))
                    .reset_index())
per_person["pct_months_ot"] = per_person["ot_months"] / per_person["months"] * 100
p12 = per_person[per_person["months"] >= MIN_MONTHS].copy()
p12["persistent"] = p12["pct_months_ot"] >= PERSISTENCE_THRESHOLD

comp_pm = pd.crosstab(pm_all["service"], pm_all["staff_group_modal"], normalize="index").reindex(index=SERVICES, columns=GROUP_ORDER) * 100
comp_p12 = pd.crosstab(p12["service"], p12["staff_group"], normalize="index").reindex(index=SERVICES, columns=GROUP_ORDER) * 100
print("% of person-months by staff group:")
display(comp_pm.round(1))
print(f"% of persons present ≥ {MIN_MONTHS} months by staff group (persistence denominator):")
display(comp_p12.round(1))

fig, ax = plt.subplots(figsize=(10, 4.2))
comp_p12.iloc[::-1].plot(kind="barh", stacked=True, ax=ax, colormap="Blues_r", edgecolor="white", width=0.7)
ax.set_xlabel(f"% of persons present at least {MIN_MONTHS} months")
ax.set_ylabel("")
ax.set_title(f"Figure EDA2-2. Staff-group composition by service ({WINDOW[0]} to {WINDOW[-1]})")
ax.set_xlim(0, 100)
ax.legend(title="Staff group", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_eda2_2_staff_group_composition.png", dpi=150)
plt.show()

In [ ]:
#4.2 KPIs BY STAFF GROUP AND SERVICE
n_group = p12.pivot_table(index="staff_group", columns="service", values="persistent", aggfunc="size").reindex(index=GROUP_ORDER, columns=SERVICES)
pers_group = p12.pivot_table(index="staff_group", columns="service", values="persistent", aggfunc="mean").reindex(index=GROUP_ORDER, columns=SERVICES) * 100
freq_group = pm_all.pivot_table(index="staff_group_modal", columns="service", values="has_ot", aggfunc="mean").reindex(index=GROUP_ORDER, columns=SERVICES) * 100
weight_base = pm_all[pm_all["has_ot"] & (pm_all["gross_pay"] > 0) & ~pm_all["flag_amount_month"]]
weight_group = weight_base.pivot_table(index="staff_group_modal", columns="service", values="ot_weight", aggfunc="median").reindex(index=GROUP_ORDER, columns=SERVICES)

print(f"Persons present ≥ {MIN_MONTHS} months (denominator):"); display(n_group)
print("Persistence rate (%):"); display(pers_group.round(1))
print("Overtime frequency (% of person-months):"); display(freq_group.round(1))
print("Overtime weight in gross pay (median %, amount-only months excluded):"); display(weight_group.round(1))
print("Rank of each staff group within its service (1 = highest persistence):")
pers_group.rank(ascending=False).astype("Int64")

In [ ]:
#4.3 DIRECTLY STANDARDISED PERSISTENCE RATE
reference = p12["staff_group"].value_counts(normalize=True).reindex(GROUP_ORDER)
standard = pd.DataFrame({
    "crude_rate_%": p12.groupby("service")["persistent"].mean().reindex(SERVICES) * 100,
    "standardised_rate_%": pers_group.mul(reference, axis=0).sum(),
})
standard["difference_pp"] = standard["standardised_rate_%"] - standard["crude_rate_%"]
standard["rank_crude"] = standard["crude_rate_%"].rank(ascending=False).astype(int)
standard["rank_standardised"] = standard["standardised_rate_%"].rank(ascending=False).astype(int)
print("Reference composition (all services, % of persons ≥ 12 months):")
print((reference * 100).round(1).to_string())
standard.round(1)

In [ ]:
#FIGURE EDA2-3: PERSISTENCE BY STAFF GROUP AND SERVICE
x = np.arange(len(GROUP_ORDER))
width = 0.8 / len(SERVICES)
fig, ax = plt.subplots(figsize=(12, 4.8))
for i, s in enumerate(SERVICES):
    bars = ax.bar(x + (i - (len(SERVICES) - 1) / 2) * width, pers_group[s], width=width, color=COLOR[s],
                  label=f"{s} (n={int(n_group[s].sum())})")
    ax.bar_label(bars, labels=[f"{v:.0f}" if pd.notna(v) else "" for v in pers_group[s]], fontsize=6.5, padding=2)
ax.set_xticks(x)
ax.set_xticklabels(GROUP_ORDER)
ax.set_xlabel("Staff group (modal over the window)")
ax.set_ylabel("% of persons present ≥ 12 months")
ax.set_title(f"Figure EDA2-3. Persons with persistent overtime (≥ {PERSISTENCE_THRESHOLD}% of their months), "
             "by staff group and service")
ax.legend(fontsize=8, title="Service (persons in base)")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_eda2_3_persistence_group_service.png", dpi=150)
plt.show()

**Observations (Section 4)**

- *Composition (Figure EDA2-2).* All services are dominated by professionals (32%–47% of persons) and physicians (19%–31%). The SSMC has more professionals and fewer physicians than the pooled network, and slightly more auxiliary and administrative staff.
- *Same ordering everywhere (Figure EDA2-3).* Auxiliary staff have the highest persistence in 5 of 6 services (administrative staff in Sur Oriente), professionals are 4th and physicians last in **all six**. The C1 finding that persistence concentrates in auxiliary and administrative staff is therefore **not specific to the SSMC**: it is a network-wide pattern.
- *Large differences within the same staff group.* Persistence among auxiliary staff goes from 22% (Oriente) to 69% (Norte, only 16 persons) and among technicians from 9% (Oriente) to 57% (Norte). Staff group alone does not determine persistence: services organise the same functions differently.
- *Standardisation.* With the pooled composition, the SSMC rate falls from 20.1% to **18.4%** (−1.6 points): part of its rate comes from having a staff mix with more high-persistence groups. Its rank (3rd) does not change. Norte rises to 27.2% because it has few auxiliary staff; its auxiliary rate rests on 16 persons and is unstable.
- *What the figures do not show:* why groups or services differ (shift design, vacancies, authorisation practice or budget are not in the payroll), nor whether the differences are statistically stable (small groups such as Norte auxiliary staff).

## 5. Distributions by service

The three variables behind the KPIs are compared with the robust measures used in C1 (median, IQR, MAD):
- overtime hours per person-month, among person-months with overtime;
- overtime weight in gross pay, among person-months with overtime and gross pay > 0 (amount-only months excluded);
- % of months with overtime per person (persons present ≥ 12 months), the variable to which the 75% threshold is applied.

In [ ]:
#5.1 DISTRIBUTION SUMMARIES BY SERVICE
def summarise(s):
    return pd.Series({"n": s.count(), "mean": s.mean(), "median": s.median(), "std": s.std(),
                      "IQR": s.quantile(0.75) - s.quantile(0.25), "MAD": (s - s.median()).abs().median(),
                      "p95": s.quantile(0.95), "max": s.max()})

with_ot = pm_all[pm_all["has_ot"]]
print("Overtime hours per person-month (person-months with overtime):")
display(with_ot.groupby("service")["ot_hours"].apply(summarise).unstack().reindex(SERVICES).round(1))
print("Overtime weight in gross pay, % (with overtime, gross pay > 0, amount-only months excluded):")
display(weight_base.groupby("service")["ot_weight"].apply(summarise).unstack().reindex(SERVICES).round(1))
print(f"% of months with overtime per person (persons present ≥ {MIN_MONTHS} months):")
p12.groupby("service")["pct_months_ot"].apply(summarise).unstack().reindex(SERVICES).round(1)

In [ ]:
#FIGURE EDA2-4: HOURS AND WEIGHT BY SERVICE
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
axes[0].boxplot([with_ot.loc[with_ot["service"] == s, "ot_hours"] for s in SERVICES], showfliers=False)
axes[0].set_title("Overtime hours per person-month\n(person-months with overtime; whiskers 1.5·IQR, outliers hidden)")
axes[0].set_ylabel("Hours paid in the month (day + night)")
axes[1].boxplot([weight_base.loc[weight_base["service"] == s, "ot_weight"] for s in SERVICES], showfliers=False)
axes[1].set_title("Overtime weight in gross pay\n(with overtime and gross pay > 0)")
axes[1].set_ylabel("% of gross pay")
for ax in axes:
    ax.set_xticks(range(1, len(SERVICES) + 1))
    ax.set_xticklabels(SERVICES)
    ax.set_xlabel("Service")
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)
fig.suptitle(f"Figure EDA2-4. Distribution of overtime hours and weight by service ({WINDOW[0]} to {WINDOW[-1]})")
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_eda2_4_hours_weight_service.png", dpi=150)
plt.show()

In [ ]:
#FIGURE EDA2-5: CUMULATIVE DISTRIBUTION OF THE % OF MONTHS WITH OVERTIME
fig, ax = plt.subplots(figsize=(10, 4.6))
for s in SERVICES:
    v = np.sort(p12.loc[p12["service"] == s, "pct_months_ot"])
    ax.step(v, np.arange(1, len(v) + 1) / len(v) * 100, where="post", color=COLOR[s], label=f"{s} (n={len(v)})")
ax.axvline(PERSISTENCE_THRESHOLD, color="black", linestyle="--", linewidth=1)
ax.text(PERSISTENCE_THRESHOLD + 1, 5, f"Threshold {PERSISTENCE_THRESHOLD}%", fontsize=9)
ax.set_xlabel("% of each person's months with overtime")
ax.set_ylabel("Cumulative % of persons")
ax.set_title(f"Figure EDA2-5. Cumulative distribution of the % of months with overtime (persons present ≥ {MIN_MONTHS} months)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_eda2_5_ecdf_persistence_service.png", dpi=150)
plt.show()

**Observations (Section 5)**

- *Hours (Figure EDA2-4, left).* Right-skewed in every service (mean above median, long tails up to 431–770 h). The SSMC has the highest median (36 h) and IQR (57 h). Sur Oriente has a very low median (13 h), but it is the service where holiday overtime, excluded by the KPI definition, is most frequent (Section 2), so its hours are understated rather than comparable.
- *Weight (Figure EDA2-4, right).* The SSMC median (14.8%) is the highest; Sur Oriente (4.8%) the lowest. Maximum values far above 100% in Occidente and Sur are the impossible values flagged in Section 3 (overtime above gross pay); they do not move the medians.
- *% of months with overtime (Figure EDA2-5).* In every service the distribution is **U-shaped**: the median person has 0%–5% of months with overtime, while the 95th percentile reaches 100% (82% in Oriente). Persistence is a property of a minority of persons who receive overtime almost every month, not a general shift of all staff. Because few persons sit near the threshold, moving it between 50% and 90% changes the level of the rate but not the ranking of services (Section 7).

## 6. Monthly series and comparison between years

- **Monthly overtime frequency** by service, with the flagged months of Section 3.2 marked.
- **Comparison between years** using the **same calendar months** (November–August) in both years, so seasonality cannot explain a difference. The second year starts after Comptroller ruling (Dictamen) E121666, applicable from 19 July 2025, which states that overtime may only be paid for work actually performed (C1 §8).
- The decline is checked **within each staff group**, to rule out a pure composition effect (Simpson's paradox).

In [ ]:
#6.1 MONTHLY OVERTIME FREQUENCY BY SERVICE
freq_month = pm_all.pivot_table(index="period", columns="service", values="has_ot", aggfunc="mean").reindex(columns=SERVICES) * 100

fig, ax = plt.subplots(figsize=(12, 4.8))
for s in SERVICES:
    ax.plot(freq_month.index, freq_month[s], marker="o", markersize=3, linewidth=1.5, color=COLOR[s], label=s)
    bad = [p for p in ATYPICAL[s] if p in freq_month.index]
    ax.scatter(bad, freq_month.loc[bad, s], s=90, facecolors="none", edgecolors="red", linewidths=1.5, zorder=3)
ax.axvline("2025-08", color="grey", linestyle=":", linewidth=1)
ax.text("2025-08", ax.get_ylim()[0] + 0.5, " first payment month after\n Dictamen E121666", fontsize=8, va="bottom")
ax.scatter([], [], s=90, facecolors="none", edgecolors="red", label="Atypical month\n(Section 3.2)")
ax.set_title("Figure EDA2-6. Monthly overtime frequency by service")
ax.set_xlabel("Payment month")
ax.set_ylabel("% of person-months with overtime")
ax.tick_params(axis="x", rotation=90)
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_eda2_6_monthly_frequency_service.png", dpi=150)
plt.show()

In [ ]:
#6.2 YEAR COMPARISON (SAME CALENDAR MONTHS), OVERALL AND BY STAFF GROUP
pm_all["year"] = np.select([pm_all["period"].between(*YEAR1), pm_all["period"].between(*YEAR2)],
                           ["Y1 Nov24-Aug25", "Y2 Nov25-Aug26"], default="other")
yy = pm_all[pm_all["year"] != "other"]

def year_summary(t):
    c = t[t["has_ot"]]
    w = c[(c["gross_pay"] > 0) & ~c["flag_amount_month"]]
    return pd.Series({"person_months": len(t), "ot_frequency_%": t["has_ot"].mean() * 100,
                      "median_hours": c["ot_hours"].median(), "median_weight_%": w["ot_weight"].median(),
                      "ot_spend_CLP_millions": t["ot_amount"].sum() / 1e6})

for label, data in {"All months": yy, "Without atypical months": yy[~yy["flag_atypical_month"]]}.items():
    print(label)
    tab = data.groupby(["service", "year"])[data.columns].apply(year_summary).unstack().reindex(SERVICES)
    tab[("ot_frequency_%", "change_pp")] = tab[("ot_frequency_%", "Y2 Nov25-Aug26")] - tab[("ot_frequency_%", "Y1 Nov24-Aug25")]
    display(tab.round(1))

by_group = (yy[~yy["flag_atypical_month"]]
            .pivot_table(index=["service", "staff_group_modal"], columns="year", values="has_ot", aggfunc="mean") * 100)
by_group["change_pp"] = by_group["Y2 Nov25-Aug26"] - by_group["Y1 Nov24-Aug25"]
change_group = by_group["change_pp"].unstack("service").reindex(index=GROUP_ORDER, columns=SERVICES)
print("Change in overtime frequency, Y2 − Y1 (percentage points), by staff group (without atypical months):")
display(change_group.round(1))
print("Staff groups with a decline, by service:", (change_group < 0).sum().to_dict())

**Observations (Section 6)**

- *Network-wide decline.* Comparing November–August of both years, overtime frequency **fell in 5 of 6 services**: SSMC −1.8 points (28.2% → 26.4%), Norte −3.8, Oriente −3.1, Sur −2.9, Sur Oriente −2.0. Occidente rises (+2.3) only because of its atypical July–August 2026; without those months it is flat (+0.2). Spending also fell in five services.
- *Within staff groups.* Without atypical months, the decline appears in all five staff groups in the SSMC, Oriente and Sur Oriente and in four groups in Sur, so it is not a composition effect there. In Norte it is concentrated in professionals (−5.6) and physicians (−3.6) while technicians rise (+7.3).
- *Revision of C1.* C1 described the SSMC decline and mentioned Dictamen E121666 as one possible explanation. The comparison shows the decline is **common to the network**, which weakens SSMC-specific explanations (for example, a local staffing change).
- *What it does not show (association, not causation).* A before/after comparison in which every service was exposed to the ruling at the same time has **no unexposed comparison group** and no random assignment, so it cannot identify the ruling's effect (A/B testing class: estimand, potential outcomes and randomisation). Budget cycles, hiring or recording changes common to all services would produce the same pattern.

## 7. Spending and sensitivity of the KPIs

- **Overtime spending** by service and staff group, and the share paid to persistent persons.
- **Threshold sensitivity** (50%, 75%, 90%).
- **KPIs with and without flagged person-months** (as C1 §6.4), to check that no flag drives the comparison.

In [ ]:
#7.1 OVERTIME SPENDING BY SERVICE
pm_all["persistent"] = pm_all.set_index(["service", "person_id"]).index.map(
    p12.set_index(["service", "person_id"])["persistent"]).fillna(False).astype(bool)
spend = pm_all.groupby("service").agg(person_months=("ot_amount", "size"), spend_CLP_millions=("ot_amount", "sum")).reindex(SERVICES)
spend["spend_CLP_millions"] /= 1e6
spend["persistent_persons_CLP_millions"] = pm_all[pm_all["persistent"]].groupby("service")["ot_amount"].sum().reindex(SERVICES) / 1e6
spend["%_paid_to_persistent"] = spend["persistent_persons_CLP_millions"] / spend["spend_CLP_millions"] * 100
spend["spend_per_person_month_CLP"] = spend["spend_CLP_millions"] * 1e6 / spend["person_months"]
spend["%_of_gross_pay"] = pm_all.groupby("service")["ot_amount"].sum() / pm_all.groupby("service")["gross_pay"].sum() * 100
display(spend.round(1))

print("Spending by staff group (% of each service's overtime spending):")
spend_group = pm_all.pivot_table(index="staff_group_modal", columns="service", values="ot_amount", aggfunc="sum").reindex(index=GROUP_ORDER, columns=SERVICES)
(spend_group / spend_group.sum() * 100).round(1)

In [ ]:
#7.2 THRESHOLD SENSITIVITY BY SERVICE
thresholds = [50, 75, 90]
sens = pd.DataFrame({f"≥{u}%": p12.groupby("service")["pct_months_ot"].apply(lambda s, u=u: (s >= u).mean() * 100)
                     for u in thresholds}).reindex(SERVICES)
sens.loc["All services"] = [(p12["pct_months_ot"] >= u).mean() * 100 for u in thresholds]
display(sens.round(1))
print("Rank of the SSMC among the six services at each threshold (1 = highest):")
sens.drop("All services").rank(ascending=False).loc["Centro"].astype(int)

In [ ]:
#7.3 KPIs WITH AND WITHOUT FLAGGED PERSON-MONTHS
flag_cols = ["flag_atypical_month", "flag_impossible", "flag_extreme_hours", "flag_zero_pay"]
scenarios = {
    "Original": pm_all,
    "Without atypical months": pm_all[~pm_all["flag_atypical_month"]],
    "Without impossible values": pm_all[~pm_all["flag_impossible"]],
    "Without extreme hours": pm_all[~pm_all["flag_extreme_hours"]],
    "Without any flag": pm_all[~pm_all[flag_cols + ["flag_amount_month"]].any(axis=1)],
}
kpi_scenarios = {name: kpis_by_service(t) for name, t in scenarios.items()}
for kpi in ["persistence_rate_%", "ot_frequency_%", "median_ot_weight_%"]:
    print(kpi)
    display(pd.DataFrame({name: t[kpi] for name, t in kpi_scenarios.items()}).T.round(1))

**Observations (Section 7)**

- *Spending.* In the common window the SSMC paid CLP 3,127 million in overtime (day + night), **72.0% of it to persistent persons**; the share ranges from 37.3% (Oriente) to 80.5% (Norte). Overtime is 3.9% of SSMC gross pay, the highest of the six services (Sur 3.4%, Sur Oriente 0.8%). Auxiliary staff receive 19.6% of SSMC overtime spending, also the highest share in the network, although they are under 7% of its persons.
- *Threshold.* The SSMC is 2nd of 6 at the 50% threshold and 3rd at 75% and 90%: its relative position does not depend on the team's choice of 75%.
- *Flags.* Removing any group of flagged person-months moves the SSMC persistence rate between 19.0% and 20.2% and the network rate between 16.1% and 16.8%. No flag changes the conclusions above.

## 8. Conclusions of the cross-service comparison

**Observations**
1. **The staff-group pattern generalises.** Auxiliary and administrative staff have the highest persistence and physicians the lowest in all six services. This strengthens the C1 finding.
2. **The SSMC's persistence level is not exceptional.** 20.1% against 16.7% for the network, 3rd of 6, and 18.4% after standardising composition. What sets the SSMC apart is **intensity**: the highest weight of overtime in gross pay, the most hours per person-month, the highest spending per person-month and the largest share of overtime spending going to auxiliary staff.
3. **The decline between years is network-wide.** It is not specific to the SSMC, and its cause cannot be identified with these data.
4. **Services differ a lot within the same staff group.** The local organisation of work matters, not only the type of function.

**Implications for W2 (Section D, representativeness)**
- The SSMC is one of six Metropolitan services, and its levels do not represent the network. Its staff-group ordering does.
- Recording practices differ between services: holiday overtime, `-` columns, position detail and incomplete Sur months. Pooling services for a model would first need harmonised definitions and a split grouped by person **across** services.

**Limitations.**
- Only fixed-term (*contrata*) staff are covered.
- Persons are identified by name within each service.
- Holiday overtime is excluded by the C1 definition.
- Sur has incomplete months.
- 22 months give one year-to-year comparison only.

**Managerial reading.** For the SSMC, the cross-service evidence moves the priority question from *"how many people have persistent overtime?"* to *"why does overtime weigh more in pay here than elsewhere, especially for auxiliary staff (drivers)?"*. Services with similar staff mix and lower intensity, such as Oriente, are natural benchmarks to ask about shift design and staffing. This is a hypothesis to investigate, not a demonstrated cause.